In [2]:
# 確認
import lightgbm as lgb

print(lgb.__version__)

4.7.0


## ライブラリ・データ読み込み

In [3]:
# ライブラリ読み込み
from pathlib import Path

import numpy as np
import pandas as pd

import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [4]:
# データ読み込み
ROOT = Path.cwd()

if ROOT.name == "smartphone_addiction":
    ROOT = ROOT.parent.parent

DATA_DIR = ROOT / "data" / "smartphone_addiction" / "raw"
OUTPUT_DIR = ROOT / "output" / "smartphone_addiction"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

train = pd.read_csv(
    DATA_DIR / "train.csv"
)

test = pd.read_csv(
    DATA_DIR / "test.csv"
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

In [5]:
TARGET = "addicted_label"

X = train.drop(
    columns=[TARGET, "id"]
).copy()

y = train[TARGET].copy()

X_test = test.drop(
    columns=["id"]
).copy()

## カテゴリ変数の処理

In [6]:
# カテゴリ列の処理
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categorical_cols)

['gender', 'stress_level', 'academic_work_impact']


C:\Users\Owner\AppData\Local\Temp\ipykernel_5720\3363954878.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [7]:
# train/test を合わせてカテゴリを統一
for col in categorical_cols:

    X[col] = (
        X[col]
        .fillna("Missing")
    )

    X_test[col] = (
        X_test[col]
        .fillna("Missing")
    )

    all_categories = pd.concat(
        [
            X[col],
            X_test[col]
        ]
    ).unique()

    X[col] = pd.Categorical(
        X[col],
        categories=all_categories
    )

    X_test[col] = pd.Categorical(
        X_test[col],
        categories=all_categories
    )

In [8]:
# 確認
X.dtypes

age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
gender                     category
stress_level               category
academic_work_impact       category
dtype: object

## LightGBM Baseline

In [9]:
lgb_params = {
    "objective": "binary",
    "n_estimators": 7000,
    "learning_rate": 0.05,

    "num_leaves": 31,
    "max_depth": -1,

    "min_child_samples": 50,

    "subsample": 0.8,
    "subsample_freq": 1,

    "colsample_bytree": 0.8,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

## 3-fold cv

In [10]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(
    len(X)
)

test_pred = np.zeros(
    len(X_test)
)

fold_scores = []
best_iterations = []

In [11]:
# cv実行
for fold, (
    train_idx,
    valid_idx
) in enumerate(
    cv.split(X, y),
    start=1
):

    print(
        f"\n===== Fold {fold} ====="
    )

    X_train_fold = X.iloc[
        train_idx
    ]

    X_valid_fold = X.iloc[
        valid_idx
    ]

    y_train_fold = y.iloc[
        train_idx
    ]

    y_valid_fold = y.iloc[
        valid_idx
    ]

    model = lgb.LGBMClassifier(
        **lgb_params
    )

    model.fit(
        X_train_fold,
        y_train_fold,

        eval_set=[
            (
                X_valid_fold,
                y_valid_fold
            )
        ],

        eval_metric="auc",

        categorical_feature=
            categorical_cols,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=200
            ),

            lgb.log_evaluation(
                period=500
            )
        ]
    )

    # validation prediction
    valid_pred = model.predict_proba(
        X_valid_fold,
        num_iteration=
            model.best_iteration_
    )[:, 1]

    oof_pred[
        valid_idx
    ] = valid_pred

    # test prediction
    test_pred += (
        model.predict_proba(
            X_test,
            num_iteration=
                model.best_iteration_
        )[:, 1]
        / cv.n_splits
    )

    # Fold AUC
    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_pred
    )

    fold_scores.append(
        fold_auc
    )

    best_iterations.append(
        model.best_iteration_
    )

    print(
        f"Fold {fold} AUC:"
        f" {fold_auc:.6f}"
    )

    print(
        "Best iteration:",
        model.best_iteration_
    )


===== Fold 1 =====


c:\Users\Owner\anaconda3\envs\kaggle_env\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[500]	valid_0's auc: 0.960203	valid_0's binary_logloss: 0.231714
[1000]	valid_0's auc: 0.962091	valid_0's binary_logloss: 0.225779
[1500]	valid_0's auc: 0.962658	valid_0's binary_logloss: 0.223854
[2000]	valid_0's auc: 0.962909	valid_0's binary_logloss: 0.222927
[2500]	valid_0's auc: 0.963059	valid_0's binary_logloss: 0.222367
[3000]	valid_0's auc: 0.963139	valid_0's binary_logloss: 0.222063
Early stopping, best iteration is:
[3266]	valid_0's auc: 0.963158	valid_0's binary_logloss: 0.221979
Fold 1 AUC: 0.963158
Best iteration: 3266

===== Fold 2 =====


c:\Users\Owner\anaconda3\envs\kaggle_env\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[500]	valid_0's auc: 0.961232	valid_0's binary_logloss: 0.229079
[1000]	valid_0's auc: 0.963021	valid_0's binary_logloss: 0.223261
[1500]	valid_0's auc: 0.9636	valid_0's binary_logloss: 0.221139
[2000]	valid_0's auc: 0.963843	valid_0's binary_logloss: 0.220176
[2500]	valid_0's auc: 0.963919	valid_0's binary_logloss: 0.219752
[3000]	valid_0's auc: 0.963963	valid_0's binary_logloss: 0.219491
Early stopping, best iteration is:
[2878]	valid_0's auc: 0.963979	valid_0's binary_logloss: 0.219471
Fold 2 AUC: 0.963979
Best iteration: 2878

===== Fold 3 =====


c:\Users\Owner\anaconda3\envs\kaggle_env\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[500]	valid_0's auc: 0.961132	valid_0's binary_logloss: 0.22938
[1000]	valid_0's auc: 0.962894	valid_0's binary_logloss: 0.223564
[1500]	valid_0's auc: 0.963432	valid_0's binary_logloss: 0.221613
[2000]	valid_0's auc: 0.963621	valid_0's binary_logloss: 0.220796
[2500]	valid_0's auc: 0.963728	valid_0's binary_logloss: 0.220306
[3000]	valid_0's auc: 0.963772	valid_0's binary_logloss: 0.220063
Early stopping, best iteration is:
[3239]	valid_0's auc: 0.963814	valid_0's binary_logloss: 0.219895
Fold 3 AUC: 0.963814
Best iteration: 3239


## 最終cv結果

In [12]:
print(
    "Fold AUCs:",
    fold_scores
)

print(
    "Mean AUC:",
    np.mean(fold_scores)
)

print(
    "Std AUC:",
    np.std(fold_scores)
)

print(
    "Best iterations:",
    best_iterations
)

Fold AUCs: [0.9631577085065067, 0.9639786415851088, 0.9638138596894518]
Mean AUC: 0.9636500699270224
Std AUC: 0.00035459187299698966
Best iterations: [3266, 2878, 3239]


In [13]:
# OOF全体でも確認
oof_auc = roc_auc_score(
    y,
    oof_pred
)

print(
    "OOF AUC:",
    oof_auc
)

OOF AUC: 0.9636470878336719


## Submission作成

In [14]:
submission = (
    sample_submission.copy()
)

submission[
    TARGET
] = test_pred

In [15]:
# チェック
assert (
    submission["id"]
    .equals(test["id"])
)

assert (
    submission[TARGET]
    .notna()
    .all()
)

assert (
    submission[TARGET]
    .between(0, 1)
    .all()
)

In [16]:
# 保存
submission_path = (
    OUTPUT_DIR
    / "submission_lightgbm.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(
    submission_path
)

c:\Users\Owner\Documents\projects\kaggle-study\output\smartphone_addiction\submission_lightgbm.csv


## CatBoostと比較

In [17]:
comparison = pd.DataFrame({
    "model": [
        "CatBoost baseline",
        "CatBoost tuned",
        "LightGBM"
    ],

    "mean_auc": [
        0.963516,
        0.963687,
        np.mean(
            fold_scores
        )
    ],

    "std_auc": [
        0.000343,
        0.000338,
        np.std(
            fold_scores
        )
    ]
})

display(
    comparison.sort_values(
        "mean_auc",
        ascending=False
    )
)

,model,mean_auc,std_auc
1,CatBoost tuned,0.963687,0.000338
2,LightGBM,0.963650,0.000355
0,CatBoost baseline,0.963516,0.000343


## OOF,test予測を保存

In [18]:
np.save(
    OUTPUT_DIR / "oof_lightgbm.npy",
    oof_pred
)

np.save(
    OUTPUT_DIR / "test_pred_lightgbm.npy",
    test_pred
)